# 05. Constructing and Serializing Trees

Rebuilding a tree from traversal outputs, or turning a tree into a string and back (**serialization**, used to store trees or send them over a network).

**Learning order:** WHY TWO TRAVERSALS → PREORDER + INORDER → INORDER + POSTORDER → SORTED ARRAY TO BST → SERIALIZE → INTERVIEW THINKING

## 1. Why one traversal is not enough

Preorder alone `[1, 2, 3]` fits many different trees. But with **inorder** we know which values are in the **left part** and which in the **right part**, so the shape becomes unique.

- **Preorder:** first element = root.
- **Postorder:** last element = root.
- **Inorder:** everything **left of the root** belongs to the left subtree, everything **right** to the right subtree.

(Values must be unique for this to work.)

In [1]:
from collections import deque

class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val, self.left, self.right = val, left, right

def tree_from_list(vals):             # level order, None = missing child
    if not vals or vals[0] is None:
        return None
    root = TreeNode(vals[0])
    q, i = deque([root]), 1
    while q and i < len(vals):
        node = q.popleft()
        if i < len(vals) and vals[i] is not None:
            node.left = TreeNode(vals[i]); q.append(node.left)
        i += 1
        if i < len(vals) and vals[i] is not None:
            node.right = TreeNode(vals[i]); q.append(node.right)
        i += 1
    return root

def level_order(root):
    if not root:
        return []
    q, res = deque([root]), []
    while q:
        res.append([])
        for _ in range(len(q)):
            node = q.popleft()
            res[-1].append(node.val)
            if node.left: q.append(node.left)
            if node.right: q.append(node.right)
    return res

def inorder(root):
    return inorder(root.left) + [root.val] + inorder(root.right) if root else []

def build_tree_pre_in(preorder, inorder):
    pos = {v: i for i, v in enumerate(inorder)}          # value -> index in inorder, O(1) lookup
    pre_idx = 0
    def build(lo, hi):                                    # build from inorder[lo..hi]
        nonlocal pre_idx
        if lo > hi:
            return None
        val = preorder[pre_idx]
        pre_idx += 1
        root = TreeNode(val)
        mid = pos[val]
        root.left = build(lo, mid - 1)                    # left first, preorder is root, left, right
        root.right = build(mid + 1, hi)
        return root
    return build(0, len(inorder) - 1)

t = build_tree_pre_in([3, 9, 20, 15, 7], [9, 3, 15, 20, 7])
print(level_order(t))   # [[3], [9, 20], [15, 7]]

[[3], [9, 20], [15, 7]]


## 2. Inorder + Postorder

Postorder is `left, right, root`, so read it **from the end**: root, then **right**, then **left**. Build the right subtree **before** the left.

In [2]:
def build_tree_in_post(inorder, postorder):
    pos = {v: i for i, v in enumerate(inorder)}
    post_idx = len(postorder) - 1
    def build(lo, hi):
        nonlocal post_idx
        if lo > hi:
            return None
        val = postorder[post_idx]
        post_idx -= 1
        root = TreeNode(val)
        mid = pos[val]
        root.right = build(mid + 1, hi)                   # right first!
        root.left = build(lo, mid - 1)
        return root
    return build(0, len(inorder) - 1)

t = build_tree_in_post([9, 3, 15, 20, 7], [9, 15, 7, 20, 3])
print(level_order(t))   # [[3], [9, 20], [15, 7]]

[[3], [9, 20], [15, 7]]


## 3. Sorted array → height-balanced BST

Pick the **middle** element as the root so both sides have nearly equal size. Recurse on the left half and right half. Height stays O(log n).

In [3]:
def sorted_array_to_bst(nums):
    def build(lo, hi):
        if lo > hi:
            return None
        mid = (lo + hi) // 2
        return TreeNode(nums[mid], build(lo, mid - 1), build(mid + 1, hi))
    return build(0, len(nums) - 1)

def height(root):
    return 0 if not root else 1 + max(height(root.left), height(root.right))

b = sorted_array_to_bst([-10, -3, 0, 5, 9])
print(inorder(b), height(b))   # [-10, -3, 0, 5, 9] 3

[-10, -3, 0, 5, 9] 3


## 4. Serialize and Deserialize (preorder with null markers)

Write preorder, but also write a marker (`#`) for every missing child. Then the string **alone** describes the tree uniquely. To rebuild, read tokens in order with the same recursion.

In [4]:
def serialize(root):
    out = []
    def dfs(node):
        if not node:
            out.append("#")
            return
        out.append(str(node.val))
        dfs(node.left)
        dfs(node.right)
    dfs(root)
    return ",".join(out)

def deserialize(data):
    tokens = iter(data.split(","))
    def build():
        tok = next(tokens)
        if tok == "#":
            return None
        node = TreeNode(int(tok))
        node.left = build()
        node.right = build()
        return node
    return build()

t = tree_from_list([1, 2, 3, None, None, 4, 5])
s = serialize(t)
print(s)                                  # 1,2,#,#,3,4,#,#,5,#,#
print(level_order(deserialize(s)))        # [[1], [2, 3], [4, 5]]

1,2,#,#,3,4,#,#,5,#,#
[[1], [2, 3], [4, 5]]


## 5. Interview thinking

**Signals:** "construct from traversals", "convert sorted array/list to BST", "serialize/deserialize", "save a tree to a file".

**Say:** "The first/last element of preorder/postorder is the root. I find it in the inorder array using a hash map for O(1), which splits left and right. Total O(n)."

**Common mistakes:** searching the inorder array linearly (O(n²)), building left before right in the postorder version, using slicing (copies, O(n²)), forgetting null markers in serialization, duplicates (ambiguous).

## Quick Revision

- Preorder root = first. Postorder root = last. Inorder splits left/right.
- Hash map `value → inorder index` for O(1).
- Postorder version: build **right** first.
- Sorted array → BST: middle as root.
- Serialize with `#` for null, preorder; deserialize with an iterator.